# PreferenceForge: Open Alignment & Direct Preference Optimization (DPO) Studio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/master72o/preferenceforge-alignment-study/blob/main/notebooks/preferenceforge_gpu_run.ipynb)
[![GitHub Repository](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/master72o/preferenceforge-alignment-study)

Welcome to the **PreferenceForge** GPU execution notebook! This notebook allows you to train and evaluate a real open-source Large Language Model using **Direct Preference Optimization (DPO)** and LoRA on a free **NVIDIA T4 GPU** (via Google Colab or Kaggle).

### Pipeline Stages:
1. **Environment Setup & GPU Verification**
2. **Load Pairwise Preference Dataset**
3. **Configure PEFT LoRA on SmolLM-135M-Instruct**
4. **Train Policy using TRL `DPOTrainer`**
5. **Compute Bradley-Terry Expected Calibration Error (ECE)**
6. **Audit Length Bias & Verbosity Exploitation**
7. **Visualize Reliability Diagram & Export Weights**

## 1. Environment & GPU Verification

In [ ]:
# Verify GPU availability
!nvidia-smi

In [ ]:
# Install lightweight training & evaluation dependencies
!pip install -q transformers datasets trl peft bitsandbytes accelerate matplotlib seaborn scipy

## 2. Load & Inspect Preference Dataset

In [ ]:
import json
import urllib.request
from datasets import Dataset

# Fetch benchmark dataset from official repository
DATASET_URL = "https://raw.githubusercontent.com/master72o/preferenceforge-alignment-study/main/data/preference_dataset.jsonl"

try:
    with urllib.request.urlopen(DATASET_URL) as response:
        raw_lines = response.read().decode('utf-8').strip().split('\n')
        raw_data = [json.loads(line) for line in raw_lines if line.strip()]
except Exception as e:
    print(f"Falling back to local data: {e}")
    with open("data/preference_dataset.jsonl", "r", encoding="utf-8") as f:
        raw_data = [json.loads(l) for l in f if l.strip()]

print(f"Loaded {len(raw_data)} preference pairs.")

# Format for TRL DPOTrainer
formatted_pairs = [
    {
        "prompt": d["prompt"],
        "chosen": d["response_chosen"],
        "rejected": d["response_rejected"],
    }
    for d in raw_data
]

dataset = Dataset.from_list(formatted_pairs)
print("Sample pair:")
print("Prompt  :", formatted_pairs[0]['prompt'])
print("Chosen  :", formatted_pairs[0]['chosen'])
print("Rejected:", formatted_pairs[0]['rejected'])

## 3. Initialize Model, Tokenizer & LoRA Adapter

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig
from trl import DPOTrainer

MODEL_ID = "HuggingFaceTB/SmolLM-135M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
print("Model and LoRA adapter successfully loaded!")

## 4. Run DPO Training Pipeline

In [ ]:
training_args = TrainingArguments(
    output_dir="./dpo_adapter_output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    max_steps=50,
    logging_steps=5,
    save_strategy="no",
    fp16=torch.cuda.is_available(),
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    beta=0.1,
    train_dataset=dataset,
    tokenizer=tokenizer,
    peft_config=lora_config,
)

print("Starting DPO Training...")
dpo_trainer.train()
dpo_trainer.save_model("./dpo_adapter_output")
print("Training complete! Adapter saved.")

## 5. Bradley-Terry Calibration & Expected Calibration Error (ECE)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

margins = [d.get("rewards_v3", {}).get("chosen", 2.5) - d.get("rewards_v3", {}).get("rejected", -1.5) for d in raw_data]
probs = [sigmoid(m) for m in margins]
labels = [1 if m > 0 else 0 for m in margins]

# Compute 10-bin ECE
n_bins = 10
bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
ece = 0.0
bin_accs, bin_confs = [], []

for i in range(n_bins):
    mask = (np.array(probs) >= bin_edges[i]) & (np.array(probs) <= bin_edges[i+1])
    if np.sum(mask) > 0:
        acc = np.mean(np.array(labels)[mask])
        conf = np.mean(np.array(probs)[mask])
        ece += (np.sum(mask) / len(probs)) * abs(acc - conf)
        bin_accs.append(acc)
        bin_confs.append(conf)

print(f"Empirical Pairwise Accuracy : {np.mean(labels)*100:.2f}%")
print(f"Expected Calibration Error   : {ece*100:.2f}%")

# Plot Reliability Diagram
plt.figure(figsize=(7, 5))
plt.plot([0, 1], [0, 1], 'k--', label="Perfect Calibration")
plt.scatter(bin_confs, bin_accs, color="#4361EE", s=80, label=f"RM v3 Checkpoint (ECE={ece*100:.1f}%)")
plt.xlabel("Mean Predicted Confidence P(Chosen > Rejected)")
plt.ylabel("Empirical Pairwise Accuracy")
plt.title("Bradley-Terry Reward Model Reliability Diagram")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6. Length Bias Diagnostics & Discordant Pair Audit

In [ ]:
delta_tokens = [len(d['response_chosen'].split()) - len(d['response_rejected'].split()) for d in raw_data]
r_val = np.corrcoef(delta_tokens, margins)[0, 1]
print(f"Length-Margin Pearson Correlation (r): {r_val:+.4f}")

plt.figure(figsize=(8, 5))
plt.scatter(delta_tokens, margins, color="#7209B7", alpha=0.7, edgecolors="k")
plt.axhline(0, color="red", linestyle="--", alpha=0.6, label="Decision Boundary (Margin = 0)")
plt.xlabel("Delta Tokens (Chosen Tokens - Rejected Tokens)")
plt.ylabel("Predicted Reward Margin (r_chosen - r_rejected)")
plt.title("Length Bias Diagnostic: Token Difference vs. Reward Margin")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()